# Projeto de MVP sobre dados do Sistema Nacional de Agravos de Notificação Obrigatória (SINAN) vs. Nexo Técnico Epidemiológico Previdenciário (NTEP)
 
Referência de estudo: https://doi.org/10.1590/0102-311X00191119 


O objetivo deste MVP é criar um pipeline de dados que permita avaliar a notificação no SUS de dados de doenças e condições de notificação obrigatória, entre acidentes de trabalho e outras condições previstas no Nexo Técnico Epidemiológico Previdenciário (NTEP).

**Contexto**
O NTEP é um cruzamento de dados de doenças e condições ocupacionais ou relacionadas ao trabalho que facilita a identificação de agravos acidentários, a partir da associação entre atividades econômicas e risco de adoecimento de trabalhadores.

O NTEP existe desde 2007, quando a Lei 11.430/2006 entrou em vigor, e é operacionalizado pelo cruzamento do CNAE (Cadastro Nacional de Atividades Econômicas) com dados de CID (Classificação Internacional de Doenças). É atualizado por meio de decreto — o mais atual é o Decreto n. 6042 de 2007. O NTEP dispensa a apresentação de Comunicado de Acidente de Trabalho para a concessão de benefícios de natureza acidentária no contexto de trabalho, que, entre outros benefícios, garantem estabilidade por até 12 meses após a volta ao trabalho e o depósito de FGTS pelo empregador.

O SINAN é um sistema de notificações geral do Ministério da Saúde, que coleta dados de doenças e condições de interesse de todos os estabelecimentos de assistência à saúde do Brasil, com regularidade substancial, variável por condição, com no máximo 60 dias de atraso da identificação da condição. As secretarias municipais e estaduais de saúde são obrigadas legalmente a apresentar dados dessa natureza ao Ministério da Saúde. A lista atualizada de condições que o SINAN registra foi definido pela PRT MS/GM 204/2016, Anexo 1.

**Objetivo principal**
Cruzar os dados do SINAN com o NTEP permite avaliar o nível de adoecimento limite geral da população e estabelecer um teto para notificações acidentais caso todas as notificações ocorressem no ambiente de trabalho.  

**Objetivo secundário**
Outra pergunta que poderia ser respondida seria se o NTEP precisa ser atualizado dado que a lista de definições do SINAN é de 2016. Por exemplo, o NTEP prevê associação de uma única arbovirose, Dengue, com alguns CNAES, mas não de outras. Dado que o vetor da doença é o mesmo e a forma de exposição seria relacionada ao trabalho, o NTEP poderia ser ajustado para incluir também Chikungunya e Zika associado às mesmas ocupações que ampliam o risco para Dengue. Os dados de Chikungunya constam no SINAN, mas não no NTEP.



# Coletando dados do **SINAN**

**Importação de forma paralelizada**

- Importação de dados .dbc do ftp datasus que arquivam documentos do SINAN desde 2008, caso estejam disponíveis (adotamos um ano de lag de implementação do sistema)
- Download apenas de tabelas com agravos de interesse do NTEP: Dengue, Chikungunya, Tuberculose, Hanseníase, Meningite, Acidentes de Trabalho, Violência sexual e outras, Hepatites. Observação 1: as condições de Chikungunya e de Zika foram adicionadas por dedução à lista de interesse — elas não estão previstas na última atualização do NTEP em 2007, embora as doenças tenham se tornado condições de interesse de saúde pública a partir de 2016 e sejam transmitidas pelo mesmo vetor da dengue, que está no NTEP. Observação 2: Foram incluídos acidente de trabalho com material biológico (ACBI), mas excluídos Acidente de trabalho sem especificação (ACGR).
- Depois, os dados foram descompactados em .dbf, parseados, categorizados e salvos em formato parquet (apenas dados das colunas de interesse, para evitar que o sistema caísse por capacidade de processamento. 

As colunas de interesse, agrupadas por camada de processamento, são:

CAMADA BRONZE: Inclui colunas presentes apenas em tabelas que envolvem acidente de trabalho e registro de Comunicação de Acidente de Trabalho (CAT)
- "CAT": categoria de acidente de trabalho

CAMADA SILVER: Inclui apenas colunas harmonizadas em todas as tabelas
- "NU_ANO": ano da notificação
- "DT_NOTIFIC": data da notificação
- "ID_MUNICIP": código de identificação do município de notificação
- "ID_MN_RESI": código de identificação do município de residência da pessoa
- "ID_AGRAVO": código do agravo
- "CID": código da Classificação Internacional de Doença
- "CS_SEXO": classificação de sexo da pessoa
- "CS_RACA": classificação da raça da pessoa

CAMADA GOLD: Inclui dados derivados
- Categoria de Capítulo de CID com as seleções de interesse do NTEP: via join com o dicionário de dados gerado antes
- Enriquecimento geográfco: Avaliação a nível de Brasil, região, estado, e município

Categorias de análise
Sexo: Feminino, Masculino, Em Branco, Ignorado
Raça: Ign/Branco, Amarela, Branca, Parta, Preta, Indigena, Ignorado
Emitia CAT: (quando este campo estiver ativo): Ign/Branco, Sim, Não, Não se aplica


**Análise dos resultados encontrados**

- Alguns dados têm defasagem de publicação: Dengue em 2026; Violência, em 2025 e 2026 (e só se tornou notificação compulsória em 2009); Hanseníase desde 2024, e Meningite e Acidente de Trabalho com Material Biológico desde 2023.
- Chikungunya e Zika só se tornaram notificação compulsória em 2015
- Tuberculose tem defasagem significativa (ausência de dados desde 2020), o que indica mudança de sistemas
- Hepatites não foram notificadas, o que indica erro de sistema e requer maior investigação

### Definição de parâmetros

Importando os pacotes necessários

In [0]:
import os
import io
import time
import struct
import tempfile
from ftplib import FTP
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date

import numpy as np
import pandas as pd
import pyreaddbc
import gc
import ctypes

Criando função de liberar memória 
Permite a devolução de memória ao sistema operacional depois de cada arquivo processado (dado que há muitos arquivos grandes em sequência, a reserva ocupa memória de forma ociosa)

In [0]:
def liberar_memoria():
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

Criação de vetores com agravos, tabelas, anos, colunas e pastas do ftp do SINAN de interesse
Evita carregamento de arquivos desnecessários

In [0]:
AGRAVOS = [
    "DENGUE", "CHIKUNGUNYA", "ZIKA", "TUBERCULOSE", "HANSEN", "MENINGITE",
    "ACIDTRAB", "VIOLENCIA", "HEPATITES",
]

MAPA_AGRAVOS_FTP = {
    "DENGUE": "DENG", "CHIKUNGUNYA": "CHIK", "ZIKA": "ZIKA", "TUBERCULOSE": "TUBE",
    "HANSEN": "HANS", "MENINGITE": "MENI", "ACIDTRAB": "ACBI", "VIOLENCIA": "VIOL",
    "HEPATITES": "HEPA",
}

ANOS = list(range(2008, date.today().year + 1))

COLUNAS_INTERESSE = [
    "NU_ANO", "DT_NOTIFIC", "ID_MUNICIP", "ID_MN_RESI", "ID_AGRAVO", "CID",
    "CS_SEXO", "CS_RACA", "CAT"
]

DIRETORIOS_SINAN = [
    "/dissemin/publicos/SINAN/DADOS/FINAIS",
    "/dissemin/publicos/SINAN/DADOS/PRELIM",
]

Criação de catálogo, esquema próprio, e volume para armazenamento da tabela final
Obs. Volume pode ser acessado tanto por park quanto por spark

In [0]:
CATALOG = "workspace"
SCHEMA = "sinan"
TABELA = f"{CATALOG}.{SCHEMA}.bronze_sinan_consolidado"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.dados_temp")

Configuração de armazenamento e paralelismo

Define onde cada tipo de dado fica guardado durante a execução (arquivos brutos, baixados do FTP, ficam num diretório temporário comum, e o resultado limpo fica em um Volume separado, para sobreviver entre sessões)

Define também quantas tarefas rodam simultaneamente em cada etapa (6 downloads do FTP do SINAN, mas apenas um arquvio processado por vez, dado que muitos têm milhões de linhas)

In [0]:
OUT_DIR = tempfile.mkdtemp(prefix="sinan_")
RAW_DIR = os.path.join(OUT_DIR, "raw")
PARQUET_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/dados_temp/parquet"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PARQUET_DIR, exist_ok=True)
print(f"Diretório de trabalho: {OUT_DIR}")

N_THREADS_DOWNLOAD = 6
N_THREADS_PARSE = 1

Definir função de listar arquivos do FTP do SINAN, tanto da pasta FINAL quanto da pasta PRELIM

In [0]:
def listar_arquivos_ftp():
    localizacao = {}
    for diretorio in DIRETORIOS_SINAN:
        ftp = FTP("ftp.datasus.gov.br", timeout=60)
        ftp.login()
        ftp.cwd(diretorio)
        for nome in ftp.nlst():
            localizacao[nome.upper()] = diretorio
        ftp.quit()
    return localizacao

Definir função que separa as tabelas e dados de interesse com o que existe no FTP 

In [0]:
def montar_tarefas(agravos, anos, mapa_ftp, localizacao_arquivos):
    tarefas = []
    for agravo in agravos:
        if agravo not in mapa_ftp:
            print(f"⊘ {agravo}: não mapeado")
            continue
        codigo_ftp = mapa_ftp[agravo]
        for ano in anos:
            ano_str = str(ano)[-2:]
            candidatos = [f"{codigo_ftp}BR{ano_str}.dbc", f"{codigo_ftp}BR{ano_str}.dbf"]
            nome_arquivo = next((c for c in candidatos if c.upper() in localizacao_arquivos), None)
            if nome_arquivo:
                diretorio = localizacao_arquivos[nome_arquivo.upper()]
                tarefas.append({"agravo": agravo, "ano": ano, "nome_arquivo": nome_arquivo, "diretorio": diretorio})
            else:
                print(f"⊘ {agravo} {ano}: não encontrado (FINAIS nem PRELIM)")
    return tarefas


Definir função que executa o download de um único arquivo para o disco local
Essa função roda em paralelo múltiplas vezes

In [0]:
def baixar_arquivo(tarefa: dict) -> dict:
    nome_arquivo = tarefa["nome_arquivo"]
    try:
        ftp = FTP("ftp.datasus.gov.br", timeout=60)
        ftp.login()
        ftp.cwd(tarefa["diretorio"])
        buffer = io.BytesIO()
        ftp.retrbinary(f"RETR {nome_arquivo}", buffer.write)
        ftp.quit()

        caminho = os.path.join(RAW_DIR, nome_arquivo)
        with open(caminho, "wb") as f:
            f.write(buffer.getvalue())

        sufixo = ".dbc" if nome_arquivo.endswith(".dbc") else ".dbf"
        return {**tarefa, "caminho_bruto": caminho, "sufixo": sufixo, "erro": None}
    except Exception as e:
        return {**tarefa, "caminho_bruto": None, "sufixo": None, "erro": str(e)[:150]}

Definir funções que: 
1. Lê só o índice do arquivo DBF (quantos registros, quais colunas), para determinar em qual posição exata (bytes) cada dado começa dentro de cada linha
e
2. Extrai as colunas de interesse do arquivo DBF em uma operação em bloco (sem ser linha por linha)![](path), para ganhar agilidade

In [0]:
def ler_cabecalho_dbf(caminho):
    with open(caminho, "rb") as f:
        cabecalho = f.read(32)
        n_registros = struct.unpack("<I", cabecalho[4:8])[0]
        tam_cabecalho = struct.unpack("<H", cabecalho[8:10])[0]
        tam_registro = struct.unpack("<H", cabecalho[10:12])[0]

        campos = []
        offset = 1
        while True:
            desc = f.read(32)
            if desc[0:1] == b"\x0d":
                break
            nome = desc[0:11].split(b"\x00")[0].decode("ascii", errors="replace")
            tamanho = desc[16]
            campos.append({"nome": nome, "tamanho": tamanho, "offset": offset})
            offset += tamanho

    return {
        "n_registros": n_registros,
        "tam_cabecalho": tam_cabecalho,
        "tam_registro": tam_registro,
        "campos": campos,
    }


def ler_dbf_colunas(caminho, colunas_desejadas, encoding="latin-1"):
    meta = ler_cabecalho_dbf(caminho)
    campos_existentes = {c["nome"]: c for c in meta["campos"]}
    colunas_presentes = [c for c in colunas_desejadas if c in campos_existentes]

    if not colunas_presentes:
        return pd.DataFrame()

    nomes = ["_flag"] + colunas_presentes
    formatos = ["S1"] + [f"S{campos_existentes[c]['tamanho']}" for c in colunas_presentes]
    offsets = [0] + [campos_existentes[c]["offset"] for c in colunas_presentes]

    dtype = np.dtype({
        "names": nomes,
        "formats": formatos,
        "offsets": offsets,
        "itemsize": meta["tam_registro"],
    })

    with open(caminho, "rb") as f:
        f.seek(meta["tam_cabecalho"])
        dados = np.fromfile(f, dtype=dtype, count=meta["n_registros"])

    df = pd.DataFrame(dados)
    df = df.loc[df["_flag"] != b"*", colunas_presentes].reset_index(drop=True)

    for col in colunas_presentes:
        df[col] = df[col].str.decode(encoding, errors="replace").str.strip()

    return df

Definir função que categoriza os dados, padronizando as variáveis dos dados brutos para variáveis com nomes padronizados autoexplicativos, trazendo rótulos para os códigos numéricos

In [0]:
def categorizar(df: pd.DataFrame) -> pd.DataFrame:
    if "NU_ANO" in df.columns:
        df["ANO_NOTIF"] = df["NU_ANO"]
    elif "DT_NOTIFIC" in df.columns:
        df["ANO_NOTIF"] = pd.to_datetime(df["DT_NOTIFIC"], errors="coerce").dt.year

    if "ID_MUNICIP" in df.columns:
        df["COD_MUN_NOTIF"] = df["ID_MUNICIP"].astype(str).str.zfill(6)
        df["COD_UF_NOTIF"] = df["COD_MUN_NOTIF"].str[:2]
        df["COD_REGIAO"] = df["COD_UF_NOTIF"].str[0]
        df["MUNICIPIO_NOTIF"] = df["COD_MUN_NOTIF"]
        df["UF_NOTIF"] = df["COD_UF_NOTIF"]
    elif "ID_MN_RESI" in df.columns:
        df["COD_MUN_NOTIF"] = df["ID_MN_RESI"].astype(str).str.zfill(6)
        df["COD_UF_NOTIF"] = df["COD_MUN_NOTIF"].str[:2]
        df["COD_REGIAO"] = df["COD_UF_NOTIF"].str[0]

    if "ID_AGRAVO" in df.columns:
        df["CID10"] = df["ID_AGRAVO"].astype(str).str.upper()
        df["CID_CAPITULO"] = df["CID10"].str[0]
    elif "CID" in df.columns:
        df["CID10"] = df["CID"].astype(str).str.upper()
        df["CID_CAPITULO"] = df["CID10"].str[0]

    if "CS_SEXO" in df.columns:
        sexo_map = {"M": "Masculino", "F": "Feminino", "I": "Ignorado"}
        df["SEXO"] = df["CS_SEXO"].map(sexo_map).fillna("Ignorado")
    else:
        df["SEXO"] = "Ignorado"

    if "CS_RACA" in df.columns:
        raca_map = {"1": "Branca", "2": "Preta", "3": "Amarela", "4": "Parda", "5": "Indígena", "9": "Ignorado"}
        df["RACA"] = df["CS_RACA"].map(raca_map).fillna("Ignorado")
    else:
        df["RACA"] = "Ignorado"

    cat_map = {"1": "Sim", "2": "Não", "3": "Não se aplica", "9": "Ign/Branco"}
    if "CAT" in df.columns:
        df["EMITIA_CAT"] = df["CAT"].map(cat_map).fillna("Ign/Branco")
    else:
        df["EMITIA_CAT"] = "Não presente"

    colunas = [
        "ANO_NOTIF", "COD_REGIAO", "COD_UF_NOTIF", "UF_NOTIF",
        "COD_MUN_NOTIF", "MUNICIPIO_NOTIF", "CID10", "CID_CAPITULO",
        "SEXO", "RACA", "EMITIA_CAT",
    ]
    return df[[c for c in colunas if c in df.columns]].copy()

Definir função "maestro" de processamento — descompactação, leitura vetorizada, categorização, salvamento de resultado como parquet, além de limpar os arquivos temporários finais

In [0]:
def processar_arquivo(download: dict) -> dict:
    if download["erro"] or not download["caminho_bruto"]:
        return {**download, "caminho_parquet": None, "erro_parse": "download falhou"}

    caminho_bruto = download["caminho_bruto"]
    caminho_dbf = caminho_bruto

    try:
        if download["sufixo"] == ".dbc":
            caminho_dbf = caminho_bruto.replace(".dbc", ".dbf")
            pyreaddbc.dbc2dbf(caminho_bruto, caminho_dbf)

        df = ler_dbf_colunas(caminho_dbf, COLUNAS_INTERESSE, encoding="latin-1")

        if df.empty:
            return {**download, "caminho_parquet": None, "erro_parse": "arquivo vazio ou sem colunas de interesse"}

        df_cat = categorizar(df)
        del df
        df_cat["AGRAVO"] = download["agravo"]

        caminho_parquet = os.path.join(PARQUET_DIR, f"{download['agravo']}_{download['ano']}.parquet")
        df_cat.to_parquet(caminho_parquet, index=False)

        return {**download, "caminho_parquet": caminho_parquet, "n_registros": len(df_cat), "erro_parse": None}
    except Exception as e:
        return {**download, "caminho_parquet": None, "erro_parse": str(e)[:150]}
    finally:
        for caminho in {caminho_bruto, caminho_dbf}:
            if caminho and os.path.exists(caminho):
                os.remove(caminho)

### Planejamento e Execução da Coleta em fases
Planejamento por etapas em cascata - que reduzem as tarefas da lista anterior
1. Mapeamento do universo disponível
2. Seleção a partir dos dados de interesse
3. Filtragem da lista a partir do que já existe no Volume

In [0]:
localizacao_arquivos = listar_arquivos_ftp()
print(f"{len(localizacao_arquivos)} arquivos listados no FTP (FINAIS + PRELIM)")

tarefas = montar_tarefas(AGRAVOS, ANOS, MAPA_AGRAVOS_FTP, localizacao_arquivos)
print(f"{len(tarefas)} arquivos a baixar")

tarefas = [
    t for t in tarefas
    if not os.path.exists(os.path.join(PARQUET_DIR, f"{t['agravo']}_{t['ano']}.parquet"))
]
print(f"{len(tarefas)} arquivos ainda faltam (pulando os já processados no Volume)")

Execução em etapas - cada uma com grau de paralelismo
1. Download de tudo em paralelo (mesmo caso falhem)
2. Processamento da fila dos arquivos menores para os maiores, liberando memória após o processamento um a um

In [0]:
t0 = time.perf_counter()
downloads = []
with ThreadPoolExecutor(max_workers=N_THREADS_DOWNLOAD) as executor:
    futures = {executor.submit(baixar_arquivo, t): t for t in tarefas}
    for future in as_completed(futures):
        r = future.result()
        downloads.append(r)
        print(f"⚠ {r['nome_arquivo']}: {r['erro']}" if r["erro"] else f"✓ {r['nome_arquivo']}")
print(f"Downloads: {time.perf_counter() - t0:.1f}s")

downloads_ok = sorted(
    [d for d in downloads if not d["erro"]],
    key=lambda d: os.path.getsize(d["caminho_bruto"]),
)

t0 = time.perf_counter()
resultados = []
with ThreadPoolExecutor(max_workers=N_THREADS_PARSE) as executor:
    futures = {executor.submit(processar_arquivo, d): d for d in downloads_ok}
    for future in as_completed(futures):
        r = future.result()
        resultados.append(r)
        liberar_memoria()
        elapsed = time.perf_counter() - t0
        if r["erro_parse"]:
            print(f"⚠ [{elapsed:.0f}s] {r['agravo']} {r['ano']}: {r['erro_parse']}")
        else:
            print(f"✓ [{elapsed:.0f}s] {r['agravo']} {r['ano']}: {r['n_registros']:,} registros")
print(f"Processamento: {time.perf_counter() - t0:.1f}s")

Consolidação: lê tudo que está no volume, substituindo a tabela definitiva de uma vez

In [0]:
if os.listdir(PARQUET_DIR):
    spark.sql(f"DROP TABLE IF EXISTS {TABELA}")
    spark.read.parquet(PARQUET_DIR).write.mode("overwrite").partitionBy("AGRAVO").saveAsTable(TABELA)
    print(f"Salvo em {TABELA}: {spark.table(TABELA).count():,} registros")
else:
    print("Nenhum dado processado com sucesso")

LIMPAR! Para quando precisar refazer a camada Bronze e fazer a camada Silver com a variável CAT correta

In [0]:
import glob
import os

CATALOG = "workspace"
SCHEMA = "sinan"
PARQUET_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/dados_temp/parquet"

removidos = []
for padrao in ["ACIDTRAB_*.parquet", "VIOLENCIA_*.parquet"]:
    for caminho in glob.glob(os.path.join(PARQUET_DIR, padrao)):
        os.remove(caminho)
        removidos.append(os.path.basename(caminho))

print(f"{len(removidos)} arquivos removidos:")
for r in removidos:
    print(" ", r)

In [0]:
import json
from pyspark.sql import functions as F

CAMINHO_JSON_NTEP = "/Volumes/workspace/sinan/dados_temp/sinan_ntep_dict.json"
TABELA_BRONZE = f"{CATALOG}.{SCHEMA}.bronze_sinan_consolidado"

with open(CAMINHO_JSON_NTEP, encoding="utf-8") as f:
    ntep_dict = json.load(f)
ntep_dict.pop("_metadata", None)


def parse_padrao_cid(codigo):
    codigo = codigo.strip()
    if "-" in codigo:
        ini, fim = codigo.split("-")
        return {"tipo": "intervalo", "letra": ini[0], "num_min": int(ini[1:3]), "num_max": int(fim[1:3])}
    if "." in codigo:
        return {"tipo": "exato", "codigo": codigo.replace(".", "")}
    return {"tipo": "prefixo", "codigo": codigo}


padroes = []
for categoria, info in ntep_dict.items():
    for codigo in info["I"]:
        p = parse_padrao_cid(codigo)
        p["categoria_ntep"] = categoria
        p["presente_no_ntep"] = info["presente_no_NTEP"]
        padroes.append(p)

PADROES_EXATO = [p for p in padroes if p["tipo"] == "exato"]
PADROES_PREFIXO = [p for p in padroes if p["tipo"] == "prefixo"]
PADROES_INTERVALO = [p for p in padroes if p["tipo"] == "intervalo"]

MAPA_CAPITULO_LETRA = {
    "A": "Capítulo I", "B": "Capítulo I", "C": "Capítulo II",
    "E": "Capítulo IV", "F": "Capítulo V", "G": "Capítulo VI",
    "I": "Capítulo IX", "J": "Capítulo X", "K": "Capítulo XI",
    "L": "Capítulo XII", "M": "Capítulo XIII", "N": "Capítulo XIV",
    "O": "Capítulo XV", "P": "Capítulo XVI", "Q": "Capítulo XVII",
    "R": "Capítulo XVIII", "S": "Capítulo XIX", "T": "Capítulo XIX",
    "V": "Capítulo XX", "W": "Capítulo XX", "X": "Capítulo XX", "Y": "Capítulo XX",
    "Z": "Capítulo XXI", "U": "Capítulo XXII",
}
MAPA_REGIAO = {"1": "Norte", "2": "Nordeste", "3": "Sudeste", "4": "Sul", "5": "Centro-Oeste"}


def construir_classificacao(cid_limpo, letra, numero, campo_resultado):
    condicoes = (
        [(cid_limpo == F.lit(p["codigo"]), p[campo_resultado]) for p in PADROES_EXATO]
        + [(F.substring(cid_limpo, 1, 3) == F.lit(p["codigo"]), p[campo_resultado]) for p in PADROES_PREFIXO]
        + [((letra == F.lit(p["letra"])) & numero.between(p["num_min"], p["num_max"]), p[campo_resultado])
           for p in PADROES_INTERVALO]
    )
    expr = F.when(condicoes[0][0], F.lit(condicoes[0][1]))
    for cond, val in condicoes[1:]:
        expr = expr.when(cond, F.lit(val))
    return expr.otherwise(F.lit(None))


df_sinan = spark.table(TABELA_BRONZE)
df_sinan = df_sinan.withColumn("CID_LIMPO", F.upper(F.regexp_replace(F.col("CID10"), r"\.", "")))
cid_limpo = F.col("CID_LIMPO")
letra = F.substring(cid_limpo, 1, 1)
numero = F.expr("try_cast(substring(CID_LIMPO, 2, 2) as int)")

mapa_capitulo = F.create_map([F.lit(x) for kv in MAPA_CAPITULO_LETRA.items() for x in kv])
mapa_regiao = F.create_map([F.lit(x) for kv in MAPA_REGIAO.items() for x in kv])

df_sinan = (
    df_sinan
    .withColumn("CATEGORIA_NTEP", construir_classificacao(cid_limpo, letra, numero, "categoria_ntep"))
    .withColumn("PRESENTE_NO_NTEP", construir_classificacao(cid_limpo, letra, numero, "presente_no_ntep"))
    .withColumn(
        "CAPITULO_CID_OFICIAL",
        F.when(letra == "D", F.when(numero <= 48, "Capítulo II").otherwise("Capítulo III"))
         .when(letra == "H", F.when(numero <= 59, "Capítulo VII").otherwise("Capítulo VIII"))
         .otherwise(mapa_capitulo[letra])
    )
    .withColumn("REGIAO_NOTIF", mapa_regiao[F.col("COD_REGIAO")])
)

df_gold_detalhe = df_sinan.filter(F.col("CATEGORIA_NTEP").isNotNull())

total_sinan = df_sinan.count()
total_gold = df_gold_detalhe.count()
print(f"Total bronze: {total_sinan:,}")
print(f"Total com CID de interesse do NTEP: {total_gold:,} ({total_gold / total_sinan * 100:.1f}%)")

DIMENSOES_BASE = [
    "ANO_NOTIF", "CATEGORIA_NTEP", "CAPITULO_CID_OFICIAL", "PRESENTE_NO_NTEP",
    "SEXO", "RACA", "EMITIA_CAT",
]

gold_brasil = df_gold_detalhe.groupBy(*DIMENSOES_BASE).count().withColumnRenamed("count", "N_NOTIFICACOES")
gold_regiao = df_gold_detalhe.groupBy(*DIMENSOES_BASE, "REGIAO_NOTIF").count().withColumnRenamed("count", "N_NOTIFICACOES")
gold_uf = df_gold_detalhe.groupBy(*DIMENSOES_BASE, "UF_NOTIF").count().withColumnRenamed("count", "N_NOTIFICACOES")

for nome, df in [
    ("gold_sinan_ntep_detalhe", df_gold_detalhe),
    ("gold_sinan_nivel_brasil", gold_brasil),
    ("gold_sinan_nivel_regiao", gold_regiao),
    ("gold_sinan_nivel_uf", gold_uf),
]:
    tabela = f"{CATALOG}.{SCHEMA}.{nome}"
    spark.sql(f"DROP TABLE IF EXISTS {tabela}")
    escritor = df.write.mode("overwrite")
    if nome == "gold_sinan_ntep_detalhe":
        escritor = escritor.partitionBy("CATEGORIA_NTEP")
    escritor.saveAsTable(tabela)
    print(f"{tabela}: {df.count():,} linhas")